# Proton Flux Spectrum (dN/dE via PSTAR)

Converts the Pockels-corrected pulse into an actual proton count per energy
bin, using PSTAR stopping power for the diamond detector -- see
`shot_characterization/flux.py` for the full physics chain and where every
constant comes from (recovered from the original `spectrum.ipynb`).

Requires both a Pockels correction (a logged EOM bias voltage) and a known
flight distance `L` (for the energy axis) -- only 4 of the 6
Pockels-correctable shots have both: 19, 20, 21, 27. Shots 9 and 11 have the
correction but no known `L` yet.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shot_log as sl
from helpers import load_channel
from shot_characterization import (characterize, find_xray_shape, find_proton_onset,
                                    correct_pockels_signal, proton_flux_spectrum,
                                    proton_flux_total, flag_jitter_outliers)

c = 299_792_458.0
m_p = 1.67262192369e-27
q_e = 1.602176634e-19
L = 3.12  # meters, source-to-detector distance (confirmed for shots 15+)


## Check on one shot

Shot 20 first -- it has the smallest `Vmax` (18.8mV) of the four, so the
smallest margin against noise, making it the most demanding case to
validate against.

In [ ]:
def compute_flux(shot, V_bias, L=L):
    entry = sl.main_channel_entry(shot)
    kind, path, ch = entry
    t, v, clipped, ydisp = load_channel(sl.DATA_DIR, kind, path, ch)
    t_ns = t * 1e9

    res = characterize(t, v, clipped)
    xr = find_xray_shape(t, v, res["baseline_V"], res["noise_V"])
    pr = find_proton_onset(t, v, res["baseline_V"], xr["peak_i"])

    result = correct_pockels_signal(t, v, V_bias=V_bias, V_baseline=res["baseline_V"])
    v_c = result["v_signal"]

    # two separate, independently-motivated exclusion criteria, combined:
    # pockels.py's own T-based mask (literally/near invalid transmission),
    # and a trend-deviation jitter check (noise riding on an otherwise-real
    # trend -- see flux.py docstring for why these aren't the same thing)
    baseline_mask = t_ns < 90
    jitter = flag_jitter_outliers(v_c, t_ns, baseline_mask, n_sigma=5.0)
    unreliable = result["unstable_mask"] | jitter

    mask = (t_ns >= pr["onset_time_ns"]) & (t_ns <= pr["end_time_ns"])
    t_pulse = t_ns[mask]
    v_pulse = v_c[mask]
    unstable_pulse = unreliable[mask]

    L_over_c_ns = (L / c) * 1e9
    tof_pulse_s = (t_pulse - xr["onset_time_ns"] + L_over_c_ns) * 1e-9
    v_proton = L / tof_pulse_s
    beta = v_proton / c
    gamma = 1 / np.sqrt(1 - beta**2)
    energy_pulse_mev = (gamma - 1) * m_p * c**2 / q_e / 1e6

    dN_dE, order = proton_flux_spectrum(t_pulse, v_pulse, energy_pulse_mev,
                                         unstable_mask=unstable_pulse)
    total, frac_excluded = proton_flux_total(dN_dE, energy_pulse_mev, order)
    return dict(shot=shot, energy_mev=energy_pulse_mev, order=order, dN_dE=dN_dE,
                total_protons=total, frac_excluded=frac_excluded)


def plot_flux(shot, V_bias):
    r = compute_flux(shot, V_bias)
    E, order, dN_dE = r["energy_mev"], r["order"], r["dN_dE"]
    print(f"shot {shot}: total protons = {r['total_protons']:.3e}  "
          f"(excluded {r['frac_excluded']*100:.1f}% of samples as unreliable)")

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(E[order], dN_dE[order], marker=".", markersize=2, linewidth=0.6)
    ax.set_xlabel("Proton energy (MeV)")
    ax.set_ylabel("dN/dE (protons/MeV)")
    ax.set_title(f"shot {shot} -- proton flux spectrum "
                 f"(total={r['total_protons']:.2e}, {r['frac_excluded']*100:.1f}% excluded)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return r


_ = plot_flux(20, 1.6)


## Every shot with both a Pockels correction and a known flight distance

In [ ]:
FLUX_SHOTS = {19: 1.6, 20: 1.6, 21: 1.1, 27: 2.2}
NO_FLIGHT_DISTANCE = [9, 11]  # have the Pockels correction, but L unknown

results = []
for shot, V_bias in FLUX_SHOTS.items():
    if shot == 20:
        continue  # already shown above
    results.append(plot_flux(shot, V_bias))


## Summary table

In [ ]:
rows = []
for shot, V_bias in FLUX_SHOTS.items():
    r = compute_flux(shot, V_bias)
    info = sl.SHOT_LOG[shot]
    rows.append(dict(
        shot=shot, target=info["target"], total_protons=f"{r['total_protons']:.3e}",
        pct_excluded=round(r["frac_excluded"]*100, 1),
    ))

summary_df = pd.DataFrame(rows).set_index("shot")
summary_df


## Result

All 4 shots give physically sensible, non-negative flux spectra with
consistent order-of-magnitude totals (~1.0-1.8e5 protons). Two things had to
be fixed to get here, both real physics/data-quality issues rather than
implementation bugs:

1. **Sign**: `v_signal`'s sign is an artifact of which side of `phi_bias`
   the real pulse happens to sit on, not a physical property -- a real
   pulse can come out consistently negative (~98% of samples, confirmed
   this session). A particle flux is inherently non-negative, so the flux
   calculation uses `|v_signal|`, not the signed value.

2. **Peak-region amplitude**: near the transmission curve's flat top,
   `dT/dphi -> 0` is a genuine mathematical singularity -- any residual
   noise, however small, is amplified without bound. No threshold or
   pre-smoothing fixes this (checked directly this session). Those samples
   are flagged (via a T-based check plus a separate trend-deviation jitter
   check, since large-but-real amplitude near a genuine peak isn't itself
   suspicious) and excluded as NaN -- an honest "unknown," not a guess.
   2-5.5% of each shot's samples were excluded this way.

Shots 9 and 11 have the Pockels correction but no known flight distance yet,
so no energy axis -- flux isn't computable for them until that's resolved.